# BÀI TẬP LỚN 03: TINH CHỈNH MẠNG SÂU PHÂN LOẠI CẢM XÚC KHÁCH HÀNG
**Học phần:** Phát triển hệ thống thông minh (PGS. TS. Trần Đình Quế)  
**Sinh viên:** Nguyễn Tuấn Tài - **MSSV:** B23DCCN731 - **Lớp:** 06  

**Mục tiêu bài thực hành:**
1. Thử nghiệm các chiến lược khởi tạo trọng số: He Normal vs Xavier/Glorot Normal vs Random Uniform.
2. Điều chỉnh cấu trúc độ sâu mạng (3 tầng vs 5 tầng vs 7 tầng).
3. Đánh giá Macro F1-Score trên dữ liệu mất cân bằng nhãn đánh giá sản phẩm.


In [ ]:
# Cell [1]: Tiền xử lý văn bản Reviews và TF-IDF 3003 chiều
import os, sys, numpy as np, pandas as pd, torch, torch.nn as nn, torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report

candidate_paths = [
    '../data/customer_reviews.csv',
    'data/customer_reviews.csv',
    '../../data/customer_reviews.csv'
]
data_path = None
for p in candidate_paths:
    if os.path.exists(p):
        data_path = p
        break

df_c = pd.read_csv(data_path)
tfidf = TfidfVectorizer(max_features=3003, stop_words='english')
X_tfidf = tfidf.fit_transform(df_c['review_text'].fillna('')).toarray()
y_label = df_c['rating'].values - 1

X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y_label, test_size=0.2, random_state=42, stratify=y_label)
print(f"X_train: {X_train.shape}, Test: {X_test.shape}")


In [ ]:
# Cell [2]: Khảo sát chiến lược khởi tạo He vs Xavier
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
        nn.init.zeros_(m.bias)

model = nn.Sequential(
    nn.Linear(3003, 128), nn.ReLU(),
    nn.Linear(128, 64),  nn.ReLU(),
    nn.Linear(64, 5)
)
model.apply(init_weights)
print("[OK] Áp dụng Kaiming/He Normalization khởi tạo trọng số thành công.")
